# 🏠 Módulo 13: El Laboratorio Virtual — Proxmox & Homelab

> **Tutor personal:** aprenderás paso a paso a virtualizar servicios en tu propio hardware usando Proxmox VE y a automatizar la gestión con Python.

**Inspirado en:** [Ardens — Homelab Series](https://www.youtube.com/@ardens_dev)

---

## 🗺️ Estructura del notebook

1. **Teoría** — Qué es Proxmox, KVM, LXC y la API REST
2. **Demo** — Cliente Python para la API (con simulación)
3. **Ejercicio TODO** — Implementa `get_storage()` y `create_snapshot()`
4. **Solución** — Implementación de referencia
5. **Quiz** — Preguntas de comprensión

## 1. 📖 Teoría

### ¿Qué es Proxmox VE?

**Proxmox Virtual Environment** es una plataforma de virtualización de código abierto basada en Debian. Permite:

| Característica | Descripción |
|---|---|
| **VMs (KVM)** | Máquinas virtuales completas con kernel propio |
| **LXC** | Contenedores Linux ligeros, comparten kernel del host |
| **Interfaz web** | Panel de control en `https://host:8006` |
| **API REST** | Todas las operaciones disponibles vía HTTP |
| **Cluster** | Múltiples nodos con alta disponibilidad |

### Arquitectura

```
Hardware (servidor físico)
└── Proxmox VE (Debian + KVM + LXC)
    ├── pve-manager   → API REST + Web UI (puerto 8006)
    ├── qemu-kvm      → Motor de virtualización
    ├── lxc           → Motor de contenedores
    └── storage
        ├── local       → /var/lib/vz
        ├── local-lvm   → LVM thin pool
        └── NFS/CEPH    → almacenamiento en red
```

### API REST de Proxmox

La API sigue el estándar REST con autenticación por **ticket + CSRF token**:

```
POST /api2/json/access/ticket  → obtener ticket
GET  /api2/json/nodes          → listar nodos
GET  /api2/json/nodes/{n}/qemu → listar VMs
GET  /api2/json/nodes/{n}/lxc  → listar contenedores
POST /api2/json/nodes/{n}/vzdump → crear backup
```

## 2. 🎬 Demo — Simulación del cliente Proxmox

Vamos a simular la API REST de Proxmox localmente para practicar sin necesitar un servidor real.

In [ ]:
# Simulador de la API REST de Proxmox
import json
from unittest.mock import patch, MagicMock

# Datos simulados del cluster
MOCK_DATA = {
    "nodes": [
        {"node": "pve1", "status": "online", "cpu": 0.12, "uptime": 432000},
        {"node": "pve2", "status": "online", "cpu": 0.05, "uptime": 259200},
    ],
    "pve1_qemu": [
        {"vmid": 100, "name": "ubuntu-server",  "status": "running", "cpus": 4, "maxmem": 4*1024**3},
        {"vmid": 101, "name": "windows-dev",    "status": "stopped", "cpus": 2, "maxmem": 8*1024**3},
        {"vmid": 102, "name": "arch-workstation","status": "running", "cpus": 6, "maxmem": 16*1024**3},
    ],
    "pve1_lxc": [
        {"vmid": 200, "name": "nginx-proxy",   "status": "running"},
        {"vmid": 201, "name": "pihole-dns",    "status": "running"},
        {"vmid": 202, "name": "gitea",         "status": "running"},
    ],
    "pve1_status": {
        "cpu": 0.12,
        "memory": {"used": 12 * 1024**3, "total": 32 * 1024**3},
        "uptime": 432000,
    }
}

def make_resp(data):
    m = MagicMock()
    m.read.return_value = json.dumps({"data": data}).encode()
    m.__enter__ = lambda s: s
    m.__exit__ = MagicMock(return_value=False)
    return m

print("✅ Datos mock cargados")
print(f"   Nodos: {[n['node'] for n in MOCK_DATA['nodes']]}")
print(f"   VMs en pve1: {[v['name'] for v in MOCK_DATA['pve1_qemu']]}")
print(f"   LXC en pve1: {[c['name'] for c in MOCK_DATA['pve1_lxc']]}")

In [ ]:
import sys, os
sys.path.insert(0, 'codigo/python')
from proxmox_client import ProxmoxClient

client = ProxmoxClient("192.168.1.10", "root@pam", "secret")

# Simular login
with patch('urllib.request.urlopen', return_value=make_resp({
    "ticket": "PVE:root@pam:DEMO",
    "CSRFPreventionToken": "csrf-demo"
})):
    client.login()
    print(f"✅ Autenticado — ticket: {client._ticket[:20]}...")

# Listar nodos
with patch('urllib.request.urlopen', return_value=make_resp(MOCK_DATA['nodes'])):
    nodos = client.get_nodes()
    print(f"\n📡 Nodos del cluster ({len(nodos)}):")
    for n in nodos:
        print(f"   • {n['node']:10} | estado: {n['status']:8} | CPU: {n['cpu']*100:.1f}%")

# Listar VMs
with patch('urllib.request.urlopen', return_value=make_resp(MOCK_DATA['pve1_qemu'])):
    vms = client.get_vms('pve1')
    print(f"\n💻 VMs en pve1 ({len(vms)}):")
    for v in vms:
        icono = '🟢' if v['status'] == 'running' else '🔴'
        print(f"   {icono} [{v['vmid']}] {v['name']:20} | {v['status']}")

# Estado del nodo
with patch('urllib.request.urlopen', return_value=make_resp(MOCK_DATA['pve1_status'])):
    status = client.get_node_status('pve1')
    mem = status['memory']
    print(f"\n📊 Recursos pve1:")
    print(f"   CPU:  {status['cpu']*100:.1f}%")
    print(f"   RAM:  {mem['used']//1024**3} GB / {mem['total']//1024**3} GB")

## 3. ✏️ Ejercicio TODO

Implementa las siguientes funciones en el cliente Proxmox:

### Función 1: `get_storage(node)`
Debe consultar el endpoint `/nodes/{node}/storage` y retornar la lista de almacenamientos.

### Función 2: `create_snapshot(node, vmid, snapname, description)`
Debe hacer un `POST` a `/nodes/{node}/qemu/{vmid}/snapshot` con los parámetros dados.

**Pista:** Los endpoints siguen el mismo patrón que `get_vms()` y `get_containers()`.

In [ ]:
# TODO: implementa las dos funciones

class ProxmoxClientExtendido(ProxmoxClient):

    def get_storage(self, node: str) -> list:
        # TODO: usa self._get() con el endpoint correcto
        raise NotImplementedError("Implementa esta función")

    def create_snapshot(self, node: str, vmid: int, snapname: str, description: str = "") -> str:
        # TODO: usa self._post() (necesitarás añadir ese método) o urllib directamente
        raise NotImplementedError("Implementa esta función")


# Prueba básica
c = ProxmoxClientExtendido("192.168.1.10", "root@pam", "secret")
c._ticket = "fake"

storage_mock = [
    {"storage": "local",      "type": "dir",  "avail": 50*1024**3},
    {"storage": "local-lvm",  "type": "lvmthin", "avail": 200*1024**3},
]

try:
    with patch('urllib.request.urlopen', return_value=make_resp(storage_mock)):
        storages = c.get_storage('pve1')
        print(f"✅ Storages: {[s['storage'] for s in storages]}")
except NotImplementedError:
    print("⏳ Pendiente: implementa get_storage()")

## 4. ✅ Solución

Aquí está la implementación de referencia:

In [ ]:
import urllib.request, urllib.parse, ssl

class ProxmoxClientCompleto(ProxmoxClient):

    def get_storage(self, node: str) -> list:
        """Lista los pools de almacenamiento de un nodo."""
        return self._get(f"/nodes/{node}/storage")

    def _post(self, path: str, data: dict) -> object:
        """Realiza un POST autenticado."""
        if not self._ticket:
            raise RuntimeError("No autenticado. Llama a login() primero.")
        url = f"{self.base_url}/{path.lstrip('/')}"
        encoded = urllib.parse.urlencode(data).encode()
        req = urllib.request.Request(url, data=encoded, method="POST")
        req.add_header("Cookie", f"PVEAuthCookie={self._ticket}")
        req.add_header("CSRFPreventionToken", self._csrf_token or "")
        with urllib.request.urlopen(req, context=self._ssl_ctx) as resp:
            import json
            return json.loads(resp.read()).get("data")

    def create_snapshot(self, node: str, vmid: int, snapname: str, description: str = "") -> str:
        """Crea un snapshot de una VM."""
        return self._post(f"/nodes/{node}/qemu/{vmid}/snapshot", {
            "snapname": snapname,
            "description": description,
        })


# --- Verificación ---
c2 = ProxmoxClientCompleto("192.168.1.10", "root@pam", "secret")
c2._ticket = "fake"
c2._csrf_token = "fake-csrf"

storage_mock = [
    {"storage": "local",     "type": "dir",     "avail": 50*1024**3},
    {"storage": "local-lvm", "type": "lvmthin", "avail": 200*1024**3},
]

with patch('urllib.request.urlopen', return_value=make_resp(storage_mock)):
    storages = c2.get_storage('pve1')
    print("Storages:")
    for s in storages:
        print(f"  • {s['storage']:15} ({s['type']:10}) — {s['avail']//1024**3} GB libres")

with patch('urllib.request.urlopen', return_value=make_resp("UPID:pve1:snap-task")):
    task = c2.create_snapshot('pve1', 100, 'snap-before-update', 'Antes de actualizar')
    print(f"\n✅ Snapshot creado — Task: {task}")

## 5. 🧠 Quiz

Responde las siguientes preguntas sin mirar el código:

### Pregunta 1
**¿Cuál es la diferencia entre una VM (KVM) y un contenedor LXC?**

<details><summary>Ver respuesta</summary>

- **VM (KVM):** emula hardware completo, tiene su propio kernel, mayor aislamiento, mayor overhead.
- **LXC:** comparte el kernel del host, más ligero y rápido, menor aislamiento, ideal para servicios Linux.

</details>

---

### Pregunta 2
**¿Por qué la API de Proxmox necesita tanto el ticket como el token CSRF?**

<details><summary>Ver respuesta</summary>

El **ticket** (cookie) autentica la identidad del usuario. El **CSRF token** previene ataques Cross-Site Request Forgery: sin él, una web maliciosa podría hacer peticiones POST en nombre del usuario usando su cookie.

</details>

---

### Pregunta 3
**¿Qué ventaja tiene un Homelab frente a usar servicios cloud para aprender?**

<details><summary>Ver respuesta</summary>

- **Costo:** pago único del hardware vs costos mensuales recurrentes.
- **Control total:** acceso a nivel de hardware, BIOS, red física.
- **Sin límites de uso:** puedes dejar VMs corriendo indefinidamente.
- **Aprendizaje real:** problemas de redes, almacenamiento y hardware que el cloud abstrae.

</details>

---

### Pregunta 4
**¿Qué parámetro de `backup_vm.sh` controla si la VM se detiene durante el backup?**

<details><summary>Ver respuesta</summary>

El parámetro `--mode` con los valores:
- `snapshot` → backup sin detener (requiere agente QEMU)
- `suspend` → pausa brevemente la VM
- `stop` → detiene completamente la VM (más consistente, más downtime)

</details>